# Facial Emotion CNN — Kaggle GPU training

Self-contained notebook (no dependency on cloning the repo) that trains a
CNN on FER2013 using CLAHE + grayscale + normalization preprocessing and
inline augmentation. Attach a FER2013 image-folder dataset via 'Add Input'
(e.g. `astraszab/facial-expression-dataset-image-folders-fer2013`, which
ships train/val/test with numeric class folders `0..6`) and enable GPU
(T4) before running.

The dataset cell auto-detects the layout: it handles numeric (`0..6`) or
emotion-named class folders, uses a provided train/test/val split when
present, and otherwise carves three **disjoint** splits out of a single
pooled folder (so the held-out test set never overlaps training data).
Class folders are mapped to the **canonical FER2013 label order**
(`angry, disgust, fear, happy, sad, surprise, neutral`) so the trained
model's outputs line up with the inference/webcam code.

Preprocessing is applied with plain NumPy/OpenCV before building the
`tf.data.Dataset` — FER2013 is small enough to fully materialize in memory,
which avoids a Keras 3 + `tf.py_function` incompatibility inside `.map()`
(`OptionalFromValue ... length 0`).

Outputs land in `/kaggle/working/artifacts/` — download `model.keras`,
`history.json`, `metrics.json`, `confusion_matrix.png`, and
`training_curves.png` into this repo's `artifacts/` folder afterward.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print('GPUs:', tf.config.list_physical_devices('GPU'))

## Constants + preprocessing (CLAHE / grayscale / normalize)

In [ ]:
IMG_SIZE = 48
NUM_CLASSES = 7
# Canonical FER2013 numeric label order: label i -> EMOTION_LABELS[i].
# Matches the original FER2013 CSV encoding and the numeric class folders
# ('0'..'6') used by the image-folder datasets.
EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
_EMOTION_TO_INDEX = {name: i for i, name in enumerate(EMOTION_LABELS)}


def fer_index_for_folder(folder_name):
    name = folder_name.strip().lower()
    if name.isdigit():
        return int(name)
    return _EMOTION_TO_INDEX[name]


def apply_clahe(gray_image, clip_limit=2.0, tile_grid_size=8):
    if gray_image.dtype != np.uint8:
        gray_image = np.clip(gray_image, 0, 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid_size, tile_grid_size))
    return clahe.apply(gray_image)


def normalize(gray_image):
    return gray_image.astype(np.float32) / 255.0

## Dataset — auto-detect the attached FER2013 layout

In [ ]:
INPUT_ROOT = Path('/kaggle/input')

NUMERIC_CLASSES = {str(i) for i in range(NUM_CLASSES)}
NAMED_CLASSES = {e.lower() for e in EMOTION_LABELS}


def is_class_dir(d):
    children = {c.name.strip().lower() for c in d.iterdir() if c.is_dir()}
    return NUMERIC_CLASSES.issubset(children) or NAMED_CLASSES.issubset(children)


def find_split_parent(root):
    """Find a dir with 'train' and 'test' subfolders that each hold the 7 FER
    class folders (numeric '0'..'6' or emotion names)."""
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        subs = {c.name.lower(): c for c in p.iterdir() if c.is_dir()}
        if 'train' in subs and 'test' in subs and is_class_dir(subs['train']) and is_class_dir(subs['test']):
            return p, subs
    return None, None


def find_single_class_parent(root):
    best, best_n = None, -1
    for p in root.rglob('*'):
        if p.is_dir() and is_class_dir(p):
            n = sum(1 for c in p.rglob('*') if c.is_file())
            if n > best_n:
                best, best_n = p, n
    return best


parent, subs = find_split_parent(INPUT_ROOT)
if parent is not None:
    train_dir = subs['train']
    test_dir = subs['test']
    val_dir = subs.get('val')
    print('Found split parent:', parent)
else:
    pool = find_single_class_parent(INPUT_ROOT)
    assert pool is not None, (
        f'Could not find FER class folders (numeric 0..6 or {sorted(NAMED_CLASSES)}) under {INPUT_ROOT}. '
        'Attach a FER2013 image-folder dataset via Add Input.'
    )
    train_dir, test_dir, val_dir = pool, None, None
    print('No train/test split found — single pool at:', pool)

print('train_dir =', train_dir)
print('test_dir  =', test_dir)
print('val_dir   =', val_dir)

## Load + preprocess into NumPy arrays, then build tf.data pipelines

In [ ]:
OUT_DIR = Path('/kaggle/working/artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 60
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1
SEED = 42
USE_CLAHE = True
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def load_split_as_arrays(directory):
    """Load every class folder under `directory`, labeling by canonical FER
    index (folder '4' -> label 4 = sad), regardless of numeric/named naming."""
    images, labels = [], []
    for cls_dir in sorted(p for p in directory.iterdir() if p.is_dir()):
        label = fer_index_for_folder(cls_dir.name)
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            if img.shape != (IMG_SIZE, IMG_SIZE):
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            if USE_CLAHE:
                img = apply_clahe(img)
            images.append(normalize(img))
            labels.append(label)
    x = np.asarray(images, dtype=np.float32).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
    y = np.asarray(labels, dtype=np.int64)
    return x, y


rng = np.random.default_rng(SEED)

if test_dir is not None:
    x_train, y_train = load_split_as_arrays(train_dir)
    x_test, y_test = load_split_as_arrays(test_dir)
    if val_dir is not None:
        x_val, y_val = load_split_as_arrays(val_dir)
    else:
        perm = rng.permutation(len(x_train))
        x_train, y_train = x_train[perm], y_train[perm]
        n_val = int(len(x_train) * VAL_SPLIT)
        x_val, y_val = x_train[:n_val], y_train[:n_val]
        x_train, y_train = x_train[n_val:], y_train[n_val:]
else:
    # Single pooled folder — carve three DISJOINT slices so test never
    # overlaps train/val (reusing the pool for both would leak).
    x_pool, y_pool = load_split_as_arrays(train_dir)
    perm = rng.permutation(len(x_pool))
    x_pool, y_pool = x_pool[perm], y_pool[perm]
    n_val = int(len(x_pool) * VAL_SPLIT)
    n_test = int(len(x_pool) * TEST_SPLIT)
    x_val, y_val = x_pool[:n_val], y_pool[:n_val]
    x_test, y_test = x_pool[n_val : n_val + n_test], y_pool[n_val : n_val + n_test]
    x_train, y_train = x_pool[n_val + n_test :], y_pool[n_val + n_test :]

class_names = list(EMOTION_LABELS)
print(f'train={len(x_train)} val={len(x_val)} test={len(x_test)}')
print('label distribution (train):', np.bincount(y_train, minlength=NUM_CLASSES).tolist())

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(min(len(x_train), 4096), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## Model (Conv2D / BatchNorm / MaxPooling / Dropout blocks + augmentation)

In [ ]:
def build_augmentation_layer():
    return tf.keras.Sequential(
        [
            layers.RandomRotation(0.08),
            layers.RandomTranslation(0.08, 0.08),
            layers.RandomZoom(0.1),
            layers.RandomFlip('horizontal'),
        ],
        name='augmentation',
    )


def conv_block(x, filters, dropout):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES, use_augmentation=True):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    if use_augmentation:
        x = build_augmentation_layer()(x)
    x = conv_block(x, 32, 0.25)
    x = conv_block(x, 64, 0.25)
    x = conv_block(x, 128, 0.30)
    x = layers.Flatten()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='facial_emotion_cnn')


model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

with open(OUT_DIR / 'history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

In [ ]:
model.save(OUT_DIR / 'model.keras')

In [ ]:
# Real held-out evaluation on the FER2013 test split
y_true, y_pred = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1).tolist())
    y_true.extend(labels.numpy().tolist())

labels_idx = list(range(NUM_CLASSES))
report = classification_report(
    y_true, y_pred, labels=labels_idx, target_names=class_names, output_dict=True, zero_division=0
)
cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

metrics = {
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'classification_report': report,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
    'epochs_trained': len(history.history['loss']),
    'batch_size': BATCH_SIZE,
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('FER2013 test confusion matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
fig.colorbar(im)
fig.tight_layout()
fig.savefig(OUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'training_curves.png', dpi=150)
plt.show()